In [1]:
# pip install xgboost
# version of xgb is 3.0.2

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sktime.datasets import load_from_tsfile_to_dataframe
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import xgboost as xgb
import json
import os

/opt/conda/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [3]:
xgb.__version__

'3.0.2'

In [4]:
def expand_and_label(x_df, y_array):
    expanded_df = pd.DataFrame(x_df['dim_0'].tolist())
    expanded_df.columns = [f'week{str(i).zfill(2)}' for i in range(expanded_df.shape[1])]
    
    expanded_df['label'] = y_array
    
    return expanded_df

def xgb_train_and_test(data_name):
    
    data_id = data_name[4:6]
    file_path_train = f"data/{data_name}/TokenItaly_vers0/TokenItaly_vers0_TRAIN.ts"
    file_path_test = f"data/{data_name}/TokenItaly_vers0/TokenItaly_vers0_TEST.ts"

    x_train, y_train = load_from_tsfile_to_dataframe(file_path_train)
    x_test, y_test = load_from_tsfile_to_dataframe(file_path_test)
    
    train_expanded = expand_and_label(x_train, y_train)
    test_expanded = expand_and_label(x_test, y_test)
    
    # separate each data into x&y and train&test
    X_train = train_expanded.drop(columns=['label'])
    y_train = train_expanded['label']
    
    X_test = test_expanded.drop(columns=['label'])
    y_test = test_expanded['label']
    
    # make them integer
    y_train = y_train.astype(int)
    y_test = y_test.astype(int)
    
    
    # define xgb model and train
    xgb_model = XGBClassifier(
        objective='binary:logistic',  # 2 classes
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
    
    xgb_model.fit(X_train, y_train)
    

    y_pred = xgb_model.predict(X_test)
    y_proba = xgb_model.predict_proba(X_test)[:, 1]


    # metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        auc = float('nan')
    
    result_dict = {
        "data_id" : data_id,
        "data_name" : data_name,
        "model" : "XGBoost",
        "Accuracy" : accuracy,
        "Precision" : precision,
        "Recall" : recall,
        "F1-score" : f1,
        "ROC-AUC score" : auc
                }

    return result_dict

In [5]:
json_path = 'results.json'
csv_path = 'results.csv'

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        all_results = json.load(f)
else:
    all_results = []

data_root_path = 'data'
data_names = sorted([
    name for name in os.listdir(data_root_path)
    if os.path.isdir(os.path.join(data_root_path, name)) and name.startswith('data')
])

for data_name in data_names:
    result_dict = xgb_train_and_test(data_name)
    all_results.append(result_dict)

with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)

df = pd.DataFrame(all_results)
df.to_csv(csv_path, index=False)